# 09 · A3 stage 1: AETHER's SatBird alignment with our RECONSTRUCTED configs (R1) and the port check (R1b)

The published AETHER code (commit 7b382c5) cannot be run as published: every alignment experiment names two
config files that are not in the repository (`PLAN.md`, A3 results log). Amendment A3.1 records our
**reconstruction** of the SatBird pair, every guessed value and its source, and the reading rule:

- a 3-seed mean "Av. (sub)" within 0.02 of the paper's 0.24 is **weak support that our set-up matches theirs**;
- a bigger gap means **"set-up uncertain, cause unknown"**;
- it is **not called a reproduction** either way.

Their code ran in its own environment (their `pyproject.toml` constraints resolved as of the commit date; the
repository has no `uv.lock`) on their Zenodo data, seeds 12345, 1 and 2. This notebook re-runs their test loop on
each seed's best checkpoint (in their environment, through `r1b_export.py`) and then:

- **R1**: reads their per-concept test index and averages it;
- **R1b**: recomputes the loss and the index with our port (`askmap.aether`), which must match within 1e-5
  (relative) and 0.005.

In [1]:
import json
import os
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from askmap.aether import concept_index, soft_contrastive_loss
from askmap.aether_us import clip_features
from askmap.config import PROJECT_ROOT, data_root

ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
VENDOR = ROOT / "vendor"
REPO = VENDOR / "aether"
REC = PROJECT_ROOT / "configs" / "aether_reconstruction"
RUN = Path(os.environ["AETHER_RUN_DIR"])  # scratch folder holding the run logs and kept checkpoints
OUT = ROOT / "aether_r1b"
SEEDS = [12345, 1, 2]
PAPER = 0.24
T_START = time.time()
print((RUN / "runs.txt").read_text())

=== seed 12345 start Wed Oct 28 19:04:29 EDT 2026
=== seed 12345 end Wed Oct 28 21:58:52 EDT 2026 exit 0
# 2026-10-29: run folder recreated here after the scratchpad copy was wiped when a session ended. Seed 12345's
# lines above are copied from the original runs.txt; its best checkpoint (best/seed_12345.ckpt) is the
# epoch_000.ckpt named as "Best ckpt path" in logs/train/runs/2026-10-28_19-05-00/train.log. Seed 1's first run
# was killed mid-training (log logs/train/runs/2026-10-28_21-59-05); seeds 1 and 2 are rerun below.
=== seed 1 start Thu Oct 29 00:47:04 EDT 2026
=== seed 1 end Thu Oct 29 03:01:24 EDT 2026 exit 0
=== seed 2 start Thu Oct 29 03:01:24 EDT 2026
=== seed 2 end Thu Oct 29 05:18:49 EDT 2026 exit 0



## Their test loop on each seed's best checkpoint (their code, their environment)

In [2]:
for seed in SEEDS:
    ckpt = RUN / "best" / f"seed_{seed}.ckpt"
    if not ckpt.exists():
        raise FileNotFoundError(f"no kept checkpoint for seed {seed}: {ckpt}")
    if (OUT / f"seed_{seed}.json").exists():
        continue
    env = {
        **os.environ,
        "PYTHONPATH": str(REPO),
        "PROJECT_ROOT": str(REPO) + "/",
        "DATA_DIR": str(VENDOR / "aether_zenodo" / "data") + "/",
        "LOGGER": "none",
        "TRAINER_PROFILE": "cpu",
    }
    r = subprocess.run(
        [
            str(REPO / ".venv" / "bin" / "python"),
            str(RUN / "r1b_export.py"),
            str(REPO),
            str(REC),
            str(ckpt),
            str(seed),
            str(OUT),
        ],
        env=env,
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-3000:])
    print(r.stdout.strip().splitlines()[-1])
META = {s: json.loads((OUT / f"seed_{s}.json").read_text()) for s in SEEDS}
NPZ = {s: np.load(OUT / f"seed_{s}.npz") for s in SEEDS}

saved /Volumes/disco2/github/climate-embeddings-crop-integrity/data/ask_the_map/aether_r1b/seed_12345.npz test metrics: 68


saved /Volumes/disco2/github/climate-embeddings-crop-integrity/data/ask_the_map/aether_r1b/seed_1.npz test metrics: 68


saved /Volumes/disco2/github/climate-embeddings-crop-integrity/data/ask_the_map/aether_r1b/seed_2.npz test metrics: 68


## R1: their test-split index per concept, "Av. (sub)" over the 25 `val_av` concepts

In [3]:
concepts = json.loads(
    (VENDOR / "aether_zenodo" / "data" / "satbird-USA-summer" / "concept_captions" / "v4.json").read_text()
)


def cname(c):
    return f"{c['col'].replace('aux_', '').replace('test-', '')}_{'max' if c['is_max'] else 'min'}"


def group(col):
    return (
        "Bioclim" if col.startswith("aux_bio") else ("Dynamic World" if col.startswith("aux_dw") else "Soil")
    )


info = pd.DataFrame(
    [
        {"name": cname(c), "col": c["col"], "val_av": c.get("val_av", True), "group": group(c["col"])}
        for c in concepts
    ]
)
rows = []
for s in SEEDS:
    m = META[s]["their_test_metrics"]
    for r in info.itertuples():
        rows.append(
            {
                "seed": s,
                "name": r.name,
                "group": r.group,
                "val_av": r.val_av,
                "index": m.get(f"test_dyn_k_index_{r.name}", np.nan),
            }
        )
R1T = pd.DataFrame(rows)
if R1T["index"].isna().any():
    raise ValueError(f"missing concepts: {R1T[R1T['index'].isna()].name.unique()}")
per_seed = R1T[R1T.val_av].groupby("seed")["index"].mean()
groups = R1T[R1T.val_av].groupby(["seed", "group"])["index"].mean().unstack()
mean3 = float(per_seed.mean())
sem3 = float(per_seed.std(ddof=1) / np.sqrt(len(per_seed)))
R1 = {
    "av_sub_per_seed": per_seed.round(4).to_dict(),
    "av_sub_mean": mean3,
    "av_sub_sem": sem3,
    "group_means_mean": groups.mean().round(4).to_dict(),
    "mean_of_group_means": float(groups.mean(axis=1).mean()),
    "test_only_concepts_mean": float(R1T[~R1T.val_av]["index"].mean()),
    "paper": PAPER,
    "paper_groups": {"Bioclim": 0.08, "Dynamic World": 0.56, "Soil": 0.03},
    "within_0_02": abs(mean3 - PAPER) <= 0.02,
}
R1["reading"] = (
    "weak support that our set-up matches theirs" if R1["within_0_02"] else "set-up uncertain, cause unknown"
)
print(json.dumps(R1, indent=1))
groups.round(3)

{
 "av_sub_per_seed": {
  "1": 0.2439,
  "2": 0.2335,
  "12345": 0.2316
 },
 "av_sub_mean": 0.2363524313045976,
 "av_sub_sem": 0.0038227364614799733,
 "group_means_mean": {
  "Bioclim": 0.0772,
  "Dynamic World": 0.5895,
  "Soil": 0.0472
 },
 "mean_of_group_means": 0.23797470211940938,
 "test_only_concepts_mean": 0.07934701597938935,
 "paper": 0.24,
 "paper_groups": {
  "Bioclim": 0.08,
  "Dynamic World": 0.56,
  "Soil": 0.03
 },
 "within_0_02": true,
 "reading": "weak support that our set-up matches theirs"
}


group,Bioclim,Dynamic World,Soil
seed,,,
1,0.095,0.594,0.029
2,0.070,0.579,0.073
12345,0.067,0.596,0.040


## R1b: our port against their code, on their data and their trained model

In [4]:
rows = []
for s in SEEDS:
    m, z = META[s], NPZ[s]
    ours_loss = float(
        soft_contrastive_loss(
            torch.as_tensor(z["loss_geo"]),
            torch.as_tensor(z["loss_text"]),
            torch.as_tensor(z["loss_aux_std"]),
            m["loss_ids"],
            torch.as_tensor(z["log_temp"]),
            m["sigma"],
        )
    )
    rel = abs(ours_loss - m["their_loss"]) / abs(m["their_loss"])
    idx_diffs = []
    for c, cfgc in enumerate(m["concept_configs"]):
        name = m["concept_names"][c]
        ours = concept_index(
            z["aux"][:, cfgc["id"]],
            z["similarity"][c],
            cfgc["theta_k"],
            cfgc["is_max"],
            m["baselines_test"][name],
        )
        idx_diffs.append(abs(ours - m["their_test_metrics"][f"test_dyn_k_index_{name}"]))
    rows.append(
        {
            "seed": s,
            "their_loss": m["their_loss"],
            "our_loss": ours_loss,
            "loss_rel_diff": rel,
            "max_index_diff": max(idx_diffs),
            "n_concepts": len(idx_diffs),
        }
    )
R1B = pd.DataFrame(rows)
R1B_PASS = bool((R1B.loss_rel_diff <= 1e-5).all() and (R1B.max_index_diff <= 0.005).all())
print("R1b PASS" if R1B_PASS else "R1b FAIL")
R1B

R1b PASS


,seed,their_loss,our_loss,loss_rel_diff,max_index_diff,n_concepts
0,12345,5.904838,5.904838,0.0,2.746091e-08,29
1,1,5.896922,5.896922,0.0,2.746091e-08,29
2,2,5.880202,5.880202,0.0,2.795437e-08,29


## Information: our CLIP text features against their encoder's concept vectors

In [5]:
from transformers import CLIPModel, CLIPTokenizer

name = "openai/clip-vit-large-patch14"
tok = CLIPTokenizer.from_pretrained(name, cache_dir=os.environ["HF_HOME"])
clip = CLIPModel.from_pretrained(name, cache_dir=os.environ["HF_HOME"])
m, z = META[SEEDS[0]], NPZ[SEEDS[0]]
ours = []
for cfgc in m["concept_configs"]:
    caps = cfgc["concept_caption"] if isinstance(cfgc["concept_caption"], list) else [cfgc["concept_caption"]]
    v = clip_features(caps, tok, clip, "cpu").mean(axis=0)
    ours.append(v / np.linalg.norm(v))
cos = np.sum(np.array(ours) * z["concept_embeds"], axis=1)
TEXT_CHECK = {"min_cosine": float(cos.min()), "mean_cosine": float(cos.mean()), "n": len(cos)}
print(TEXT_CHECK)

/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1029 05:35:12.593000 44123 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 20620.79it/s]

{'min_cosine': 0.9999997615814209, 'mean_cosine': 1.0, 'n': 29}


In [6]:
V = {"R1": R1, "R1b": {"pass": R1B_PASS, "table": R1B.to_dict(orient="records")}, "text_check": TEXT_CHECK}
(RES / "09_verdicts.json").write_text(json.dumps(V, indent=1, default=float))
R1T.to_csv(RES / "09_r1_concepts.csv", index=False)
print(f"Run time {(time.time() - T_START) / 60:.1f} min")

Run time 16.3 min
